# MouserCV - Notebook 1: Evaluate SAM3 Segmentation

This notebook evaluates **SAM3** (Segment Anything Model 3) from Ultralytics
for mouse segmentation and tracking in behavioral videos.

**Workflow:**
1. Download video from GCS
2. Extract frames
3. Prompt SAM3 on the first frame
4. Propagate masks across all frames
5. Evaluate quality and upload results

**Runtime:** Select *GPU* (T4 or better) under Runtime > Change runtime type.

## 1. Setup

In [ ]:
!pip install -q ultralytics google-cloud-storage opencv-python-headless matplotlib pycocotools

## 2. Authenticate to Google Cloud

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated successfully.')

## 3. Configuration

Edit the parameters below for your video.

In [ ]:
# ============================================================
# EDITABLE PARAMETERS
# ============================================================
BUCKET = "mousercv-data"
VIDEO_ID = "test_001"
PROJECT = "demo"
NUM_MICE = 2  # Number of mice to track

# SAM3 prompt boxes — one [x1, y1, x2, y2] per mouse.
# Adjust after viewing frame 0.
PROMPT_BOXES = [
    [100, 150, 300, 350],  # Mouse 1
    [400, 200, 600, 400],  # Mouse 2
]

assert len(PROMPT_BOXES) == NUM_MICE, "Provide one prompt box per mouse"

## 4. Download Video from GCS

In [ ]:
import os
from google.cloud import storage

client = storage.Client()
bucket = client.bucket(BUCKET)

gcs_video_path = f"videos/{PROJECT}/{VIDEO_ID}.mp4"
local_video_path = f"/tmp/{VIDEO_ID}.mp4"

blob = bucket.blob(gcs_video_path)
blob.download_to_filename(local_video_path)
file_size_mb = os.path.getsize(local_video_path) / (1024 * 1024)
print(f"Downloaded {gcs_video_path} -> {local_video_path} ({file_size_mb:.1f} MB)")

## 5. Extract Frames

In [ ]:
import cv2
import os

frames_dir = f"/tmp/frames/{VIDEO_ID}"
os.makedirs(frames_dir, exist_ok=True)

cap = cv2.VideoCapture(local_video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video: {width}x{height} @ {fps:.1f} fps, {frame_count} frames, "
      f"{frame_count / fps:.1f} sec")

idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_path = os.path.join(frames_dir, f"{idx:05d}.jpg")
    cv2.imwrite(frame_path, frame)
    idx += 1
cap.release()
print(f"Extracted {idx} frames to {frames_dir}")

## 6. Load SAM3 Model

In [ ]:
from ultralytics import SAM

model = SAM("sam3.pt")
print(f"SAM3 model loaded: {model.info()}")

## 7. Prompt SAM3 on First Frame

Display frame 0 and run initial segmentation with the configured bounding-box prompts.
If the masks look wrong, go back to the Config cell and adjust `PROMPT_BOXES`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load frame 0
frame0_path = os.path.join(frames_dir, "00000.jpg")
frame0 = cv2.imread(frame0_path)
frame0_rgb = cv2.cvtColor(frame0, cv2.COLOR_BGR2RGB)

# Display frame 0 with prompt boxes
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(frame0_rgb)
colors = ["#FF0000", "#00FF00", "#0000FF", "#FFFF00", "#FF00FF"]
for i, box in enumerate(PROMPT_BOXES):
    x1, y1, x2, y2 = box
    rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                         linewidth=2, edgecolor=colors[i % len(colors)],
                         facecolor="none")
    ax.add_patch(rect)
    ax.text(x1, y1 - 5, f"Mouse {i + 1}", color=colors[i % len(colors)],
            fontsize=12, fontweight="bold")
ax.set_title("Frame 0 with prompt boxes")
ax.axis("off")
plt.tight_layout()
plt.show()

# Run SAM3 prediction on frame 0
results_frame0 = model.predict(
    frame0_path,
    bboxes=PROMPT_BOXES,
    verbose=False,
)
print(f"Frame 0: detected {len(results_frame0[0].masks)} masks")

## 8. Run Video Propagation

Process all frames, propagating masks from the initial prompt.
This collects per-frame mask data including RLE encoding, bounding boxes, and centroids.

In [ ]:
import json
from pycocotools import mask as mask_utils


def encode_mask_rle(binary_mask: np.ndarray) -> str:
    """Encode a binary mask to COCO RLE format (JSON-safe string)."""
    # pycocotools expects Fortran-order uint8 array
    fortran_mask = np.asfortranarray(binary_mask.astype(np.uint8))
    rle = mask_utils.encode(fortran_mask)
    rle["counts"] = rle["counts"].decode("utf-8")
    return json.dumps(rle)


def compute_mask_metrics(binary_mask: np.ndarray):
    """Compute area, centroid, and bounding box from a binary mask."""
    ys, xs = np.where(binary_mask > 0)
    if len(xs) == 0:
        return None
    area = int(len(xs))
    cx, cy = float(np.mean(xs)), float(np.mean(ys))
    x1, y1 = int(np.min(xs)), int(np.min(ys))
    x2, y2 = int(np.max(xs)), int(np.max(ys))
    return {
        "area": area,
        "centroid_x": cx,
        "centroid_y": cy,
        "bbox": [x1, y1, x2, y2],
    }


# Propagate through all frames
all_mask_records = []
frame_paths = sorted(
    [os.path.join(frames_dir, f) for f in os.listdir(frames_dir) if f.endswith(".jpg")]
)

print(f"Processing {len(frame_paths)} frames...")
results = model.track(
    source=local_video_path,
    bboxes=PROMPT_BOXES,
    verbose=False,
    stream=True,
)

for frame_idx, result in enumerate(results):
    if result.masks is None:
        continue
    for obj_id, mask_tensor in enumerate(result.masks.data):
        binary_mask = mask_tensor.cpu().numpy().astype(np.uint8)
        metrics = compute_mask_metrics(binary_mask)
        if metrics is None:
            continue
        record = {
            "frame": frame_idx,
            "object_id": obj_id,
            "mask_rle": encode_mask_rle(binary_mask),
            "bbox": metrics["bbox"],
            "centroid_x": metrics["centroid_x"],
            "centroid_y": metrics["centroid_y"],
            "area": metrics["area"],
            "confidence": float(result.boxes.conf[obj_id]) if result.boxes is not None else 1.0,
        }
        all_mask_records.append(record)
    if (frame_idx + 1) % 100 == 0:
        print(f"  Processed {frame_idx + 1} frames...")

print(f"\nTotal mask records: {len(all_mask_records)}")

## 9. Visualize Mask Overlays

In [ ]:
def overlay_masks_on_frame(frame_rgb, frame_idx, records, alpha=0.4):
    """Overlay segmentation masks on a frame image."""
    overlay = frame_rgb.copy()
    mask_colors = [
        (255, 0, 0), (0, 255, 0), (0, 0, 255),
        (255, 255, 0), (255, 0, 255),
    ]
    frame_records = [r for r in records if r["frame"] == frame_idx]
    for rec in frame_records:
        rle = json.loads(rec["mask_rle"])
        binary = mask_utils.decode(rle)
        color = mask_colors[rec["object_id"] % len(mask_colors)]
        for c in range(3):
            overlay[:, :, c] = np.where(
                binary > 0,
                overlay[:, :, c] * (1 - alpha) + color[c] * alpha,
                overlay[:, :, c],
            )
    return overlay.astype(np.uint8)


# Select 5 evenly-spaced frames
total_frames_processed = max(r["frame"] for r in all_mask_records) + 1
sample_indices = np.linspace(0, total_frames_processed - 1, 5, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, fidx in enumerate(sample_indices):
    fpath = os.path.join(frames_dir, f"{fidx:05d}.jpg")
    img = cv2.cvtColor(cv2.imread(fpath), cv2.COLOR_BGR2RGB)
    overlaid = overlay_masks_on_frame(img, fidx, all_mask_records)
    axes[i].imshow(overlaid)
    axes[i].set_title(f"Frame {fidx}")
    axes[i].axis("off")

# Hide the 6th subplot
axes[5].axis("off")
fig.suptitle("SAM3 Mask Overlays (5 evenly-spaced frames)", fontsize=14)
plt.tight_layout()
plt.show()

## 10. Quality Metrics

In [ ]:
import collections

# Per-object statistics
obj_frames = collections.defaultdict(set)
obj_confidences = collections.defaultdict(list)

for rec in all_mask_records:
    oid = rec["object_id"]
    obj_frames[oid].add(rec["frame"])
    obj_confidences[oid].append(rec["confidence"])

print("=" * 60)
print("SAM3 Quality Report")
print("=" * 60)
print(f"Total frames in video:    {frame_count}")
print(f"Total mask records:       {len(all_mask_records)}")
print()

all_detected_frames = set()
for oid in sorted(obj_frames.keys()):
    n_frames = len(obj_frames[oid])
    mean_conf = np.mean(obj_confidences[oid])
    coverage = n_frames / frame_count * 100
    all_detected_frames.update(obj_frames[oid])
    print(f"Object {oid}:")
    print(f"  Frames with mask:     {n_frames}/{frame_count} ({coverage:.1f}%)")
    print(f"  Mean confidence:      {mean_conf:.3f}")
    print(f"  Min confidence:       {min(obj_confidences[oid]):.3f}")
    print()

dropped = frame_count - len(all_detected_frames)
print(f"Frames with no masks:   {dropped} ({dropped / frame_count * 100:.1f}%)")

## 11. Export to JSONL

In [ ]:
import json

results_dir = f"/tmp/results/{VIDEO_ID}"
os.makedirs(results_dir, exist_ok=True)
masks_path = os.path.join(results_dir, "masks.jsonl")

with open(masks_path, "w") as f:
    for record in all_mask_records:
        f.write(json.dumps(record) + "\n")

file_size_mb = os.path.getsize(masks_path) / (1024 * 1024)
print(f"Wrote {len(all_mask_records)} records to {masks_path} ({file_size_mb:.1f} MB)")

## 12. Upload Results to GCS

In [ ]:
gcs_masks_path = f"results/{VIDEO_ID}/masks.jsonl"
blob = bucket.blob(gcs_masks_path)
blob.upload_from_filename(masks_path)
print(f"Uploaded {masks_path} -> gs://{BUCKET}/{gcs_masks_path}")
print("\nDone! SAM3 evaluation complete.")